# BIL545 · 5. Hafta · Colab defteri: birliktelik analizi

Veri Madenciliğinde İleri Konular · Dr. Selma Güzel · Üsküdar Üniversitesi

Bu defterde, konuların sırasıyla: **sık öğe kümeleri**, **destek** ve **güven**, **Apriori** (aday üretme ve budama), **kapalı ve maksimal** sık öğe kümeleri, **birliktelik kuralları**, **FP-Growth**, gerçek veride (Weka'daki `supermarket.arff`) kural madenciliği, kuralların **ilginçliği** (lift, ki-kare, null işlemler ve null-değişmez ölçütler) ve **Proje 1, 3. bölümün** ikinci yarısı (A ve B seçenekleri).

Bu haftanın kodlarında en çok kullanılan araç `mlxtend` kütüphanesidir: `apriori`, `fpgrowth` ve `association_rules`. Her adımda **hangi işlemlerin sayıldığını** ara adımlarla göstereceğiz; bir destek ya da güven değerini önce elle, sonra kütüphaneyle hesaplayacağız.

**Okuma:** Han, Kamber ve Pei, *Data Mining: Concepts and Techniques*, 3. baskı, 6. bölüm: 6.1 (temel kavramlar), 6.2.1 (Apriori), 6.2.2 (sık öğe kümelerinden kural üretme), 6.2.4 (FP-Growth), 6.3 (ilginçlik ölçütleri). **Proje 1, 3. bölümün makalesi:** Wu, T., Chen, Y. ve Han, J. (2010). Re-examination of interestingness measures in pattern mining: A unified framework. *Data Mining and Knowledge Discovery*, 21(3), 371–397.

**Bu defter nasıl kullanılır?**

* Hücreleri **yukarıdan aşağıya, sırayla** çalıştırın: hücreye tıklayıp **Shift + Enter**. Sonraki hücreler öncekilerde tanımlanan değişkenleri kullanır; ortadan başlarsanız `NameError` hatası alırsınız. Defteri baştan çalıştırmak için: Çalışma zamanı → Tümünü çalıştır.
* Kodlar **satır satır ya da birkaç satırlık parçalar hâlinde** verilmiştir. Her parçanın önünde iki açıklama vardır:
  * 🐍 **Python:** satırdaki Python kavramı (Python bilmeniz gerekmez).
  * 📊 **Konu:** satırın veri madenciliği açısından ne yaptığı ve neden yapıldığı.
* 🔍 **Ara adım** hücreleri, bir hesabın neye uygulandığını görebilmeniz için ara sonucu gösterir (ör. bir desteğin hangi işlemlerden sayıldığı).
* Her bölümün sonunda kodun **sunumdaki hâli** tek hücre olarak tekrar verilir (slayttaki kodun aynısı); çıktısı sunumdakiyle aynı olmalıdır. Sunumda karşılığı olmayan bölümlerde bu hücrenin adı **toplu hâli**dir. ✅ **Sonucu okuyalım** kutusu sonucu yorumlar.
* Kodda `#` işaretinden sonraki her şey **yorumdur**: Python onu çalıştırmaz, insanlar için yazılmış nottur.

### Bu haftanın sözlüğü

| Terim | Tanım |
|---|---|
| **Öğe** (item) | Bir işlemde geçebilen tek şey: bir ürün, bir tanı, bir sayfa. |
| **İşlem** (transaction) | Birlikte gerçekleşen öğelerin kümesi: bir sepet, bir sipariş, bir oturum. |
| **Öğe kümesi** (itemset) | Öğelerden oluşan bir küme; *k* öğeli olana *k*-öğe kümesi denir. |
| **Destek** (support) | Bir öğe kümesini içeren işlemlerin oranı. |
| **Sık öğe kümesi** | Desteği en az destek eşiğine (`min_support`) eşit ya da büyük olan öğe kümesi. |
| **Birliktelik kuralı** | A ⇒ B biçiminde bir ifade: "A'yı içeren işlemler B'yi de içerme eğiliminde." A ve B ortak öğesi olmayan öğe kümeleridir. |
| **Güven** (confidence) | A'yı içeren işlemlerden kaçının B'yi de içerdiği. |

Formüller (N: toplam işlem sayısı):

destek(A ⇒ B) = (A ve B'yi birlikte içeren işlem sayısı) / N

güven(A ⇒ B) = destek(A ∪ B) / destek(A)

## 0. Kurulum

**Konu.** `mlxtend`, sık örüntü madenciliği için hazır işlevler içeren bir Python kütüphanesidir. Colab'da kurulu olabilir ama sürümü eski olabilir; işlevlerin yazımı sürümler arasında değiştiği için defterin hazırlandığı sürümü kuruyoruz.

🐍 **Python:** Satır başındaki `!` işareti, bu satırın Python değil, **sistem komutu** olduğunu söyler. `pip install` bir kütüphaneyi kurar; `-q` (*quiet*) kurulum mesajlarını kısaltır; `==0.25.0` belirli bir sürümü ister; `--no-deps` (*no dependencies*) yalnız bu kütüphaneyi kurar, onun **bağımlılıklarına** (çalışmak için kullandığı başka kütüphanelere: numpy, pandas, scipy) dokunmaz.

In [1]:
!pip install -q --no-deps mlxtend==0.25.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.0 MB/s eta 0:00:00


⚠️ **Kurulum notu: derste aldığımız `_slice` hatası**

Bu defterin ilk sürümünde kurulum satırı `!pip install -q mlxtend==0.25.0` idi. mlxtend 0.25.0, kendi bağımlılıklarının (numpy, pandas, scipy, scikit-learn) Colab'dakilerden **daha yeni** sürümlerini istiyor. pip bu yüzden numpy'yi, Colab oturumu açıkken güncelledi. Oturumun belleğinde eski numpy yüklüydü, diskte ise yeni numpy vardı; Python ikisini karıştırınca bir sonraki hücredeki `import` şu hatayı verdi:

`ImportError: cannot import name '_slice' from 'numpy._core.umath'`

Yani hata bizim kodumuzda değil, **kurulumdaydı**. Şimdiki satırdaki `--no-deps`, yalnız mlxtend'i kuruyor ve Colab'ın kendi numpy, pandas ve scipy sürümlerine dokunmuyor. Defterin bütün kodları bu şekilde, Colab'dakine benzer eski sürümlerle de denendi; çıktılar aynı.

**Hâlâ bu hatayı görüyorsanız** (ör. eski defteri çalıştırdıysanız): Çalışma zamanı → **Oturumu yeniden başlat** (Runtime → Restart session), sonra hücreleri yeniden yukarıdan aşağıya çalıştırın.

**Genel ders:** bir kütüphaneyi kurduktan sonra Colab "oturumu yeniden başlatın" uyarısı verirse ya da bir `import` beklenmedik biçimde hata verirse, ilk deneyeceğiniz şey oturumu yeniden başlatmaktır. Projelerinizde de kurulum hücresini defterin **en başına** koyun ve kurulumdan önce başka bir hücre çalıştırmayın.

🐍 **Python:**
* `import pandas as pd`: tablo kütüphanesi pandas'ı `pd` takma adıyla açar (2. haftadan tanıdık).
* `from mlxtend.preprocessing import TransactionEncoder`: `mlxtend`'in `preprocessing` (ön işleme) bölümünden yalnız `TransactionEncoder`'ı alır.
* `from mlxtend.frequent_patterns import ...`: sık örüntü bölümünden dört işlev alır; virgülle ayrılır.

📊 **Konu:** `TransactionEncoder` işlem listesini tabloya çevirir; `apriori` ve `fpgrowth` sık öğe kümelerini bulur; `fpmax` maksimal sık öğe kümelerini bulur; `association_rules` sık öğe kümelerinden kural üretir.

In [2]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, fpmax, association_rules

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


🔍 **Ara adım:** Kurulu sürümü kontrol edelim; `0.25.0` görmelisiniz. `mlxtend.__version__` kütüphanenin sürüm bilgisini tutan bir değişkendir.

In [3]:
import mlxtend
print(mlxtend.__version__)

0.25.0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## 1. Küçük bir sepet örneği: işlemler, destek ve güven

**Konu.** Elle izlenebilecek kadar küçük bir veriyle başlıyoruz: bir marketin 9 işlemi (sepeti), 5 öğe. Bu veri, ders kitabındaki Apriori örneğinin (Han, Kamber ve Pei, Tablo 6.1, T100–T900) aynısıdır; yalnız öğe adları Türkçeleştirildi: I1 = ekmek, I2 = peynir, I3 = süt, I4 = yağ, I5 = zeytin. Böylece kitaptaki Şekil 6.2 ile adım adım karşılaştırabilirsiniz.

🐍 **Python:** **Liste listesi.** Köşeli parantez bir **liste**dir: sıralı bir değerler dizisi. Burada her eleman da bir liste: dıştaki liste 9 işlemi, içteki her liste bir işlemin öğelerini tutar. Metinler tırnak içinde yazılır. Komut, parantez kapanana kadar birden çok satıra yayılabilir.

📊 **Konu:** İşlem verisinin en doğal hâli: her işlem, içindeki öğelerin listesi. İşlemlerin uzunlukları farklıdır (2 ile 4 öğe).

In [4]:
islemler = [
    ["ekmek", "peynir", "zeytin"],          # T100
    ["peynir", "yağ"],                      # T200
    ["peynir", "süt"],                      # T300
    ["ekmek", "peynir", "yağ"],             # T400
    ["ekmek", "süt"],                       # T500
    ["peynir", "süt"],                      # T600
    ["ekmek", "süt"],                       # T700
    ["ekmek", "peynir", "süt", "zeytin"],   # T800
    ["ekmek", "peynir", "süt"],             # T900
]

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔍 **Ara adım:** `len` bir listenin eleman sayısını verir: 9 işlem. `islemler[0]` listenin **ilk** elemanıdır (Python saymaya 0'dan başlar), `islemler[7]` sekizincisi (T800).

In [5]:
print(len(islemler))
print(islemler[0])
print(islemler[7])

9
['ekmek', 'peynir', 'zeytin']
['ekmek', 'peynir', 'süt', 'zeytin']


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:** `TransactionEncoder()` bir **nesne** oluşturur ve `te` adına atanır. Nesne, hem veri hem de o veriyle çalışan işlevler (yöntemler) taşıyan bir yapıdır; yöntemler nokta ile çağrılır.
* `te.fit(islemler)`: işlemlerdeki **bütün farklı öğeleri** öğrenir (sütun adları olacak).
* `.transform(islemler)`: her işlemi bir True/False satırına çevirir.
* `te.columns_`: öğrenilen öğe adları (alfabetik sırada).
* `pd.DataFrame(..., columns=...)`: sonucu sütun adlarıyla bir tabloya çevirir; tablo `X` adına atanır.

📊 **Konu:** **İşlem tablosu** (tek-sıcak kodlama, 3. hafta): her satır bir işlem, her sütun bir öğe; hücre True ise öğe o işlemde var. `mlxtend`'in işlevleri veriyi bu biçimde ister.

In [6]:
te = TransactionEncoder()
X = pd.DataFrame(te.fit(islemler).transform(islemler), columns=te.columns_)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔍 **Ara adım:** Tabloyu 1/0 olarak görelim: `astype(int)` True'yu 1, False'u 0 yapar. İlk satır T100: ekmek, peynir, zeytin.

In [7]:
X.astype(int)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,ekmek,peynir,süt,yağ,zeytin
0,1,1,0,0,1
1,0,1,0,1,0
2,0,1,1,0,0
3,1,1,0,1,0
4,1,0,1,0,0
5,0,1,1,0,0
6,1,0,1,0,0
7,1,1,1,0,1
8,1,1,1,0,0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


🐍 **Python:** True/False bir sütunda `sum()` True'ları **sayar** (True = 1, False = 0).

📊 **Konu:** Her öğenin **destek sayısı**: kaç işlemde geçtiği. Kitaptaki C1 tablosunun aynısı.

In [8]:
print(X.sum())

ekmek     6
peynir    7
süt       6
yağ       2
zeytin    2
dtype: int64


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

### Destek: elle

🐍 **Python:**
* `X[["ekmek", "peynir"]]`: iki sütunu birlikte seçer (çift köşeli parantez: içteki bir liste).
* `.all(axis=1)`: her **satırda**, seçilen sütunların **hepsi** True mu? (`axis=1`: satır boyunca). Sonuç her işlem için tek bir True/False.

📊 **Konu:** {ekmek, peynir} öğe kümesini içeren işlemler: ikisi birlikte olan sepetler.

In [9]:
ikisi_de = X[["ekmek", "peynir"]].all(axis=1)
print(ikisi_de.tolist())

[True, False, False, True, False, False, False, True, True]


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:** `sum()` True'ları sayar; `mean()` True'ların **oranını** verir (True sayısı ÷ satır sayısı). `round(…, 3)` üç basamağa yuvarlar.

📊 **Konu:** {ekmek, peynir} 4 işlemde geçiyor (T100, T400, T800, T900): destek = 4 / 9 = 0,444.

In [10]:
print(ikisi_de.sum(), round(ikisi_de.mean(), 3))

4 0.444


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

### Güven: elle

📊 **Konu:** güven(ekmek ⇒ peynir) = destek({ekmek, peynir}) / destek({ekmek}). Ekmekli 6 sepetin 4'ünde peynir de var: 4 / 6 = 0,667. **Ters yön** farklıdır: güven(peynir ⇒ ekmek) = 4 / 7 = 0,571. Destek simetriktir, güven değildir.

🐍 **Python:** `X["ekmek"].sum()` ekmekli işlem sayısı; `/` bölme.

In [11]:
print(round(ikisi_de.sum() / X["ekmek"].sum(), 3))    # ekmek ⇒ peynir
print(round(ikisi_de.sum() / X["peynir"].sum(), 3))   # peynir ⇒ ekmek

0.667
0.571


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Sunumdaki hâli** (tek hücre; slayttaki kodun aynısı):

In [12]:
!pip install -q --no-deps mlxtend==0.25.0
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, fpmax, association_rules

islemler = [["ekmek", "peynir", "zeytin"], ["peynir", "yağ"], ["peynir", "süt"],
            ["ekmek", "peynir", "yağ"], ["ekmek", "süt"], ["peynir", "süt"],
            ["ekmek", "süt"], ["ekmek", "peynir", "süt", "zeytin"], ["ekmek", "peynir", "süt"]]
te = TransactionEncoder()
X = pd.DataFrame(te.fit(islemler).transform(islemler), columns=te.columns_)
print(X.sum())                                         # öğe destek sayıları

ikisi_de = X[["ekmek", "peynir"]].all(axis=1)          # {ekmek, peynir} içeren işlemler
print(ikisi_de.sum(), round(ikisi_de.mean(), 3))       # destek sayısı ve destek
print(round(ikisi_de.sum() / X["ekmek"].sum(), 3))     # güven(ekmek ⇒ peynir)
print(round(ikisi_de.sum() / X["peynir"].sum(), 3))    # güven(peynir ⇒ ekmek)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

ekmek     6
peynir    7
süt       6
yağ       2
zeytin    2
dtype: int64
4 0.444
0.667
0.571


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

✅ **Sonucu okuyalım:** Öğe destekleri ekmek 6, peynir 7, süt 6, yağ 2, zeytin 2. {ekmek, peynir}: 4 işlem, destek **0,444**. güven(ekmek ⇒ peynir) **0,667**, güven(peynir ⇒ ekmek) **0,571**. Bir kural "güçlü" sayılmak için iki eşiği birden geçmelidir: en az destek (`min_support`) ve en az güven (`min_confidence`).

## 2. Sık öğe kümeleri ve Apriori

**Konu.** 5 öğeden 2⁵ − 1 = 31 boş olmayan öğe kümesi kurulabilir; 100 öğede bu sayı 10³⁰'u aşar. Hepsini saymak olanaksızdır. **Apriori özelliği** işi küçültür:

**Sık bir öğe kümesinin bütün alt kümeleri de sıktır.** Eşdeğeri: bir öğe kümesi sık değilse, onu içeren hiçbir küme sık olamaz.

Apriori bu yüzden **düzey düzey** çalışır: önce 1 öğeli sık kümeleri (L1) bulur; onlardan 2 öğeli **adayları** (C2) üretir, sayar, sık olanları (L2) tutar; L2'den C3'ü üretir, … Her düzeyde iki adım vardır: **birleştirme** (aday üretme) ve **budama** (alt kümelerinden biri sık olmayan adayı saymadan atma).

Bu bölümde en az destek sayısı 2 işlem: `min_support` = 0,2 (9 işlemde 0,2 × 9 = 1,8, yani en az 2 işlem; 2 / 9 = 0,222 ≥ 0,2). 4. haftadaki **buzdağı küpü** de aynı fikirdir: yalnız eşiği aşan hücreleri saklamak.

🐍 **Python:** `apriori(X, min_support=0.2, use_colnames=True)`: `X` tablosunda desteği en az 0,2 olan bütün öğe kümelerini bulur. `use_colnames=True`, sonuçta sütun numaraları yerine **öğe adlarını** kullanır. Sonuç bir tablodur: `support` (destek) ve `itemsets` (öğe kümesi) sütunları.

📊 **Konu:** Bütün sık öğe kümeleri (L1, L2, L3, …) tek seferde.

In [13]:
sık = apriori(X, min_support=0.2, use_colnames=True)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔍 **Ara adım:** Sonucu olduğu gibi görelim. `itemsets` sütunundaki `frozenset({...})`, **değiştirilemeyen küme** demektir: sırası olmayan, tekrarsız bir öğe topluluğu. Okuması zor; bir sonraki adımda okunur hâle getireceğiz.

In [14]:
sık

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,support,itemsets
0,0.666667,(ekmek)
1,0.777778,(peynir)
2,0.666667,(süt)
3,0.222222,(yağ)
4,0.222222,(zeytin)
5,0.444444,"(ekmek, peynir)"
6,0.444444,"(ekmek, süt)"
7,0.222222,"(ekmek, zeytin)"
8,0.444444,"(peynir, süt)"
9,0.222222,"(peynir, yağ)"


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:**
* `lambda s: ", ".join(sorted(s))`: **adsız, tek satırlık bir işlev**. "Bir `s` al; öğelerini alfabetik sırala (`sorted`); aralarına virgül koyarak tek bir metne birleştir (`", ".join`)" demektir.
* `def`/`lambda` ile tanımlanan işlev bir değişkene atanabilir: burada adı `yaz`.

📊 **Konu:** Öğe kümelerini `ekmek, peynir` biçiminde yazdırmak için küçük bir yardımcı.

In [15]:
yaz = lambda s: ", ".join(sorted(s))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔍 **Ara adım:** Yardımcıyı tek bir kümeye uygulayalım. `frozenset([...])` listeden bir küme kurar.

In [16]:
print(yaz(frozenset(["zeytin", "ekmek"])))

ekmek, zeytin


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:** `sık["itemsets"].apply(yaz)`: `apply`, verilen işlevi sütunun **her değerine ayrı ayrı** uygular. Sonuç yeni bir sütun olarak (`kume`) eklenir. `.apply(len)`: her kümenin eleman sayısı (`uzunluk`).

📊 **Konu:** Her sık öğe kümesinin kaç öğeli olduğu: hangi düzeyde (L1, L2, L3) bulunduğu.

In [17]:
sık["kume"] = sık["itemsets"].apply(yaz)
sık["uzunluk"] = sık["itemsets"].apply(len)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:** `round(3)` destekleri üç basamağa yuvarlar; `sık[["kume", "support"]]` iki sütunu seçer.

In [18]:
print(sık[["kume", "support"]].round(3))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

                     kume  support
0                   ekmek    0.667
1                  peynir    0.778
2                     süt    0.667
3                     yağ    0.222
4                  zeytin    0.222
5           ekmek, peynir    0.444
6              ekmek, süt    0.444
7           ekmek, zeytin    0.222
8             peynir, süt    0.444
9             peynir, yağ    0.222
10         peynir, zeytin    0.222
11     ekmek, peynir, süt    0.222
12  ekmek, peynir, zeytin    0.222


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔍 **Ara adım:** Her düzeyde kaç sık öğe kümesi var? `value_counts()` her değerden kaç tane olduğunu sayar; `sort_index()` sonucu 1, 2, 3 sırasına koyar.

In [19]:
print(sık["uzunluk"].value_counts().sort_index())

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

uzunluk
1    5
2    6
3    2
Name: count, dtype: int64


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

### Apriori özelliği: aday üretme ve budama (elle)

**Konu.** L2'den C3'ü üretelim ve budayalım. Kütüphane bunu içeride yapıyor; burada görünür kılıyoruz.

🐍 **Python:** **Liste üreteci:** `[set(s) for s in ...]` köşeli parantez içinde bir döngüdür: "uzunluğu 2 olan her `s` kümesini al, `set` (değiştirilebilir küme) yap, hepsini bir listede topla."
* `sık[sık["uzunluk"] == 2]`: **mantıksal filtre** (4. hafta): yalnız uzunluğu 2 olan satırlar.

📊 **Konu:** L2: 2 öğeli sık kümeler.

In [20]:
L2 = [set(s) for s in sık[sık["uzunluk"] == 2]["itemsets"]]

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [21]:
print(L2)

[{'ekmek', 'peynir'}, {'ekmek', 'süt'}, {'ekmek', 'zeytin'}, {'peynir', 'süt'}, {'peynir', 'yağ'}, {'peynir', 'zeytin'}]


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:**
* `adaylar = []`: **boş bir liste**; döngü içinde doldurulacak.
* **İç içe döngü:** L2'deki her `a` kümesi için, L2'deki her `b` kümesine bakılır.
* `a | b`: iki kümenin **birleşimi** (∪).
* `if len(u) == 3 and u not in adaylar:` iki koşul `and` ile: birleşim 3 öğeliyse **ve** listede henüz yoksa… (`not in`: içinde değil mi?)
* `adaylar.append(u)`: listenin sonuna ekler. Hangi satırın hangi bloğa ait olduğunu **girinti** belirler.

📊 **Konu:** **Birleştirme adımı:** iki 2 öğeli sık kümenin birleşimi 3 öğeli ise bir C3 adayıdır.

In [22]:
adaylar = []
for a in L2:
    for b in L2:
        u = a | b
        if len(u) == 3 and u not in adaylar:
            adaylar.append(u)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔍 **Ara adım:** Kaç aday üretildi, hangileri? `sorted(adaylar, key=yaz)`: kümeleri yazılışlarına göre sıralar; böylece çıktı her çalıştırmada aynı sırada gelir (kümelerin kendi sırası yoktur).

In [23]:
print(len(adaylar))
for u in sorted(adaylar, key=yaz):
    print(yaz(u))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

7
ekmek, peynir, süt
ekmek, peynir, yağ
ekmek, peynir, zeytin
ekmek, süt, zeytin
peynir, süt, yağ
peynir, süt, zeytin
peynir, yağ, zeytin


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:**
* `from itertools import combinations`: `combinations(küme, 2)` kümenin **2 öğeli bütün alt kümelerini** verir (4. haftadaki küboit kafesinde kullandık).
* `all(...)`: içindeki koşulların **hepsi** doğruysa True verir.
* `set(alt) in L2`: bu alt küme L2'de var mı?
* Liste üreteci, bu kez bir **koşulla**: `[u for u in adaylar if ...]`: koşulu sağlayan adayları tut.

📊 **Konu:** **Budama adımı:** bir adayın 2 öğeli alt kümelerinden biri bile L2'de yoksa (sık değilse), Apriori özelliğine göre aday da sık olamaz; **veriyi taramadan** atılır.

In [24]:
from itertools import combinations
budanmis = [u for u in adaylar if all(set(alt) in L2 for alt in combinations(u, 2))]

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔍 **Ara adım:** Bir adayın neden atıldığını görelim: {ekmek, süt, zeytin} adayının alt kümelerinden {süt, zeytin} L2'de yok (süt ve zeytin yalnız T800'de birlikte: 1 işlem).

In [25]:
aday = {"ekmek", "süt", "zeytin"}
for alt in combinations(sorted(aday), 2):
    print(alt, set(alt) in L2)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

('ekmek', 'süt') True
('ekmek', 'zeytin') True
('süt', 'zeytin') False


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:** Kalan her aday için `X[list(u)].all(axis=1).sum()`: adayın bütün öğelerini içeren işlem sayısı (1. bölümdeki kalıp). `list(u)`: küme sütun seçimi için listeye çevrilir.

📊 **Konu:** Veriyi **yalnız budamadan kalan adaylar için** tarıyoruz: destek sayısı 2 ve üstü olanlar L3'tür.

In [26]:
for u in sorted(budanmis, key=yaz):
    print(yaz(u), X[list(u)].all(axis=1).sum())

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

ekmek, peynir, süt 2
ekmek, peynir, zeytin 2


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Sunumdaki hâli** (tek hücre; slayttaki kodun aynısı):

In [27]:
sık = apriori(X, min_support=0.2, use_colnames=True)      # en az 2 / 9 işlem
yaz = lambda s: ", ".join(sorted(s))
sık["kume"] = sık["itemsets"].apply(yaz)
sık["uzunluk"] = sık["itemsets"].apply(len)
print(sık[["kume", "support"]].round(3))
print(sık["uzunluk"].value_counts().sort_index())            # L1, L2, L3 boyutları

L2 = [set(s) for s in sık[sık["uzunluk"] == 2]["itemsets"]]
adaylar = []                                                 # birleştirme: C3
for a in L2:
    for b in L2:
        u = a | b
        if len(u) == 3 and u not in adaylar:
            adaylar.append(u)
from itertools import combinations
budanmis = [u for u in adaylar if all(set(alt) in L2 for alt in combinations(u, 2))]
print(len(adaylar), "aday,", len(budanmis), "budamadan sonra")
for u in sorted(budanmis, key=yaz):
    print(yaz(u), X[list(u)].all(axis=1).sum())             # yalnız bunlar sayılır

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

                     kume  support
0                   ekmek    0.667
1                  peynir    0.778
2                     süt    0.667
3                     yağ    0.222
4                  zeytin    0.222
5           ekmek, peynir    0.444
6              ekmek, süt    0.444
7           ekmek, zeytin    0.222
8             peynir, süt    0.444
9             peynir, yağ    0.222
10         peynir, zeytin    0.222
11     ekmek, peynir, süt    0.222
12  ekmek, peynir, zeytin    0.222
uzunluk
1    5
2    6
3    2
Name: count, dtype: int64
7 aday, 2 budamadan sonra
ekmek, peynir, süt 2


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

ekmek, peynir, zeytin 2


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

✅ **Sonucu okuyalım:** 13 sık öğe kümesi: L1'de 5, L2'de 6, L3'te 2. Birleştirme 7 aday üretti; budama 5'ini **veriye bakmadan** attı; kalan iki aday ({ekmek, peynir, süt} ve {ekmek, peynir, zeytin}) 2'şer işlemde geçiyor, ikisi de sık. Kitaptaki birleştirme kuralı (ilk k − 2 öğesi aynı olan kümeleri birleştirmek) 6 aday üretir; bizim basit "birleşimi 3 öğeli olsun" kuralımız {ekmek, peynir, yağ} adayını da üretti. Budamadan sonra sonuç aynıdır.

Apriori'nin iki maliyeti: çok sayıda aday üretmek ve **her düzeyde veriyi yeniden taramak**. FP-Growth (5. bölüm) bu ikisini de ortadan kaldırır.

## 3. Kapalı ve maksimal sık öğe kümeleri

**Konu.** Sık bir kümenin bütün alt kümeleri de sık olduğu için, uzun bir sık küme çok sayıda sık alt küme getirir (100 öğeli bir küme 2¹⁰⁰ − 1). İki **sıkıştırılmış gösterim**:

* **Kapalı sık öğe kümesi:** sıktır ve **aynı desteğe sahip** hiçbir üst kümesi yoktur. Kapalı kümeler, bütün sık kümeleri **destekleriyle birlikte** geri kurmaya yeter (kayıpsız).
* **Maksimal sık öğe kümesi:** sıktır ve **hiçbir üst kümesi sık değildir**. Hangi kümelerin sık olduğunu söyler ama desteklerini kaybeder (kayıplı).

Her maksimal küme kapalıdır; tersi doğru değildir.

🐍 **Python:** **Sözlük üreteci:** `{anahtar: değer for ...}`. `zip` iki sütunu yan yana dolaşır: her sık küme (`s`) anahtar, desteği (`d`) değer olur. Sonuç: "küme → destek" eşlemesi.

📊 **Konu:** Her sık kümenin desteğine hızla bakabilmek için bir tablo.

In [28]:
destek = {s: d for s, d in zip(sık["itemsets"], sık["support"])}

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:** Kümelerde `s < t`: "`s`, `t`'nin **öz alt kümesi** mi?" (`t` daha büyük ve `s`'nin bütün öğelerini içeriyor). `any(...)`: içindeki koşullardan **en az biri** doğruysa True.

📊 **Konu:**
* **kapalı:** aynı desteğe sahip bir üst kümesi olmayanlar,
* **maksimal:** hiç sık üst kümesi olmayanlar.

In [29]:
kapali = [s for s in destek if not any(s < t and destek[t] == destek[s] for t in destek)]
maksimal = [s for s in destek if not any(s < t for t in destek)]

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔍 **Ara adım:** {ekmek, zeytin} neden kapalı değil? Desteği 2 / 9; üst kümesi {ekmek, peynir, zeytin}'in desteği de 2 / 9. Zeytinli iki sepetin ikisinde de ekmek ve peynir var; {ekmek, zeytin}'i ayrıca saklamak yeni bilgi vermez.

In [30]:
print(round(destek[frozenset(["ekmek", "zeytin"])], 3),
      round(destek[frozenset(["ekmek", "peynir", "zeytin"])], 3))

0.222 0.222


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:** `fpmax`, maksimal sık kümeleri doğrudan bulan `mlxtend` işlevidir (FP-Growth'un bir türü). Kendi listemizle karşılaştırıyoruz: `set(...) == set(...)` iki koleksiyonun **aynı elemanlara** sahip olup olmadığını sorar (sıra önemsiz).

In [31]:
mak = fpmax(X, min_support=0.2, use_colnames=True)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Toplu hâli** (tek hücre):

In [32]:
destek = {s: d for s, d in zip(sık["itemsets"], sık["support"])}
kapali = [s for s in destek if not any(s < t and destek[t] == destek[s] for t in destek)]
maksimal = [s for s in destek if not any(s < t for t in destek)]
print(len(destek), "sık,", len(kapali), "kapalı,", len(maksimal), "maksimal")
print("Kapalı:", [yaz(s) for s in kapali])
print("Maksimal:", [yaz(s) for s in maksimal])

mak = fpmax(X, min_support=0.2, use_colnames=True)
print(set(mak["itemsets"]) == set(maksimal))                 # kütüphaneyle aynı mı?

13 sık, 9 kapalı, 3 maksimal
Kapalı: ['ekmek', 'peynir', 'süt', 'ekmek, peynir', 'ekmek, süt', 'peynir, süt', 'peynir, yağ', 'ekmek, peynir, süt', 'ekmek, peynir, zeytin']
Maksimal: ['peynir, yağ', 'ekmek, peynir, süt', 'ekmek, peynir, zeytin']
True


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

✅ **Sonucu okuyalım:** 13 sık kümeden **9**'u kapalı, **3**'ü maksimal. Maksimal kümeler: {peynir, yağ}, {ekmek, peynir, süt}, {ekmek, peynir, zeytin}. Bu üç küme ve alt kümeleri, sık kümelerin tamamıdır; ama örneğin {ekmek}'in desteğinin 6 / 9 olduğunu maksimal kümelerden çıkaramazsınız. Kapalı kümelerden çıkarabilirsiniz: bir kümenin desteği, onu içeren kapalı kümelerin en büyük desteğidir. `fpmax` aynı üç kümeyi buldu (`True`).

Neden önemli: büyük veride sık küme sayısı on binlere çıkar (6. bölümde göreceğiz); kapalı ya da maksimal kümeler raporlamayı mümkün kılar.

## 4. Sık öğe kümelerinden birliktelik kuralları

**Konu.** Kural üretimi iki adımdır: (1) bütün sık öğe kümelerini bul (Apriori ya da FP-Growth), (2) her sık küme Z için, Z'nin boş olmayan her **öz alt kümesi** A'dan bir kural kur: A ⇒ Z − A. Güveni hesapla, en az güveni geçenleri tut. Kuralın desteği Z'nin desteğidir; zaten sık olduğu için destek eşiğini otomatik geçer.

Önce elle, tek bir küme için: Z = {ekmek, peynir, zeytin}.

🐍 **Python:**
* `Z = frozenset([...])`: üç öğeli küme.
* Dış döngü `k = 1, 2`: öncülün (A) öğe sayısı. `range(1, len(Z))` 1'den başlar, `len(Z)` = 3'ü dâhil etmez.
* İç döngü: Z'nin k öğeli bütün alt kümeleri (`combinations`).
* `oncul = frozenset(alt)`: alt küme, kuralın **öncülü** (A) olur.
* `Z - oncul`: küme **farkı**: Z'de olup öncülde olmayanlar (kuralın sonuç kısmı, B).
* `destek[Z] / destek[oncul]`: güven formülü; destekleri 3. bölümdeki sözlükten okuyoruz (öncül de sık olduğu için sözlükte var).

📊 **Konu:** Z'den üretilebilecek bütün kurallar ve güvenleri.

In [33]:
Z = frozenset(["ekmek", "peynir", "zeytin"])
for k in range(1, len(Z)):
    for alt in combinations(sorted(Z), k):
        oncul = frozenset(alt)
        print(yaz(oncul), "⇒", yaz(Z - oncul), round(destek[Z] / destek[oncul], 3))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

ekmek ⇒ peynir, zeytin 0.333
peynir ⇒ ekmek, zeytin 0.286
zeytin ⇒ ekmek, peynir 1.0
ekmek, peynir ⇒ zeytin 0.5
ekmek, zeytin ⇒ peynir 1.0
peynir, zeytin ⇒ ekmek 1.0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔍 **Ara adım:** Bir güveni elle doğrulayalım: güven(ekmek ⇒ peynir, zeytin) = destek(Z) / destek({ekmek}) = (2 / 9) / (6 / 9) = 2 / 6 = 0,333.

In [34]:
print(round(2 / 6, 3))

0.333


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:** `association_rules(sık, metric="confidence", min_threshold=0.7)`: sık kümeler tablosundan bütün kuralları üretir; `metric` hangi ölçüte göre süzüleceğini, `min_threshold` eşiği söyler. Sonuç bir tablodur: her satır bir kural.

📊 **Konu:** En az güveni %70 olan bütün kurallar, 13 sık kümenin hepsinden.

In [35]:
kurallar = association_rules(sık, metric="confidence", min_threshold=0.7)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔍 **Ara adım:** Tabloda hangi sütunlar var? `antecedents` **öncül** (A), `consequents` **sonuç** (B); `support`, `confidence` ve `lift` bu haftanın ölçütleri; geri kalanlar (`leverage`, `conviction`, `kulczynski`, …) başka ilginçlik ölçütleri. `kulczynski`'ye 7. bölümde döneceğiz.

In [36]:
print(kurallar.columns.tolist())

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

['antecedents', 'consequents', 'antecedent support', 'consequent support', 'support', 'confidence', 'lift', 'representativity', 'leverage', 'conviction', 'zhangs_metric', 'jaccard', 'certainty', 'kulczynski']


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:** Okunur sütunlar ekleyip (öncül `A`, sonuç `B`) dört sütunu yazdırıyoruz. `sort_values(["A", "B"])` tabloyu önce `A`'ya, eşitlerde `B`'ye göre sıralar (kuralların üretilme sırası her çalıştırmada değişebilir; sıralayınca çıktı hep aynı olur). `to_string(index=False)` satır numaralarını yazmaz.

In [37]:
kurallar["A"] = kurallar["antecedents"].apply(yaz)
kurallar["B"] = kurallar["consequents"].apply(yaz)
kurallar = kurallar.sort_values(["A", "B"])
print(kurallar[["A", "B", "support", "confidence"]].round(3).to_string(index=False))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

             A             B  support  confidence
 ekmek, zeytin        peynir    0.222         1.0
peynir, zeytin         ekmek    0.222         1.0
           yağ        peynir    0.222         1.0
        zeytin         ekmek    0.222         1.0
        zeytin ekmek, peynir    0.222         1.0
        zeytin        peynir    0.222         1.0

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

**Sunumdaki hâli** (tek hücre; slayttaki kodun aynısı):

In [38]:
destek = {s: d for s, d in zip(sık["itemsets"], sık["support"])}
Z = frozenset(["ekmek", "peynir", "zeytin"])                 # elle: tek bir sık küme
for k in range(1, len(Z)):
    for alt in combinations(sorted(Z), k):
        oncul = frozenset(alt)
        print(yaz(oncul), "⇒", yaz(Z - oncul), round(destek[Z] / destek[oncul], 3))

kurallar = association_rules(sık, metric="confidence", min_threshold=0.7)
kurallar["A"] = kurallar["antecedents"].apply(yaz)
kurallar["B"] = kurallar["consequents"].apply(yaz)
kurallar = kurallar.sort_values(["A", "B"])
print(kurallar[["A", "B", "support", "confidence"]].round(3).to_string(index=False))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

ekmek ⇒ peynir, zeytin 0.333
peynir ⇒ ekmek, zeytin 0.286
zeytin ⇒ ekmek, peynir 1.0
ekmek, peynir ⇒ zeytin 0.5
ekmek, zeytin ⇒ peynir 1.0
peynir, zeytin ⇒ ekmek 1.0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

             A             B  support  confidence
 ekmek, zeytin        peynir    0.222         1.0
peynir, zeytin         ekmek    0.222         1.0
           yağ        peynir    0.222         1.0
        zeytin         ekmek    0.222         1.0
        zeytin ekmek, peynir    0.222         1.0
        zeytin        peynir    0.222         1.0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

✅ **Sonucu okuyalım:** Z = {ekmek, peynir, zeytin}'den 6 kural çıkar; güvenleri 0,333 ile 1,0 arasında. %70 eşiğini üçü geçiyor: zeytin ⇒ ekmek, peynir; ekmek, zeytin ⇒ peynir; peynir, zeytin ⇒ ekmek (hepsi 1,0). Bütün sık kümelerden %70 güvenle **6** kural çıkıyor; hepsinin güveni 1,0 ve hepsi seyrek öğelerden (zeytin, yağ) başlıyor: zeytin ya da yağ alan her sepette ekmek ya da peynir de var.

**Dikkat:** Güveni 1,0 olan bir kural, yalnız 2 işleme dayanıyor. Küçük destekli kuralların güveni oynaktır (4. haftadaki küçük hücrelerdeki oranlar gibi). Güven tek başına bir kuralın ilginç olduğunu da söylemez: 7. bölümün konusu.

## 5. FP-Growth: aday üretmeden

**Konu.** FP-Growth (*frequent pattern growth*), veriyi **iki kez** tarar ve aday üretmez:

1. **İlk tarama:** öğe desteklerini say; sık olmayan öğeleri at; sık öğeleri **desteğe göre azalan** sıraya diz (L sırası).
2. **İkinci tarama:** her işlemin öğelerini L sırasına dizip bir **FP-ağacına** (sık örüntü ağacı) ekle. Ortak **önekleri** (baştaki öğeleri) aynı olan işlemler ağaçta aynı yolu paylaşır; düğümlerde sayaçlar tutulur. Ağaç, verinin sıkıştırılmış bir kopyasıdır.
3. **Madencilik:** her öğe için, o öğeyle biten yolları (koşullu örüntü tabanı) toplayıp küçük bir **koşullu FP-ağacı** kur ve aynı işlemi bu küçük ağaçta özyinelemeli tekrarla.

Kitap bu yüzden "böl ve fethet" der: büyük bir arama, öğe başına küçük aramalara bölünür. Burada 1. ve 2. adımın girdisini görüyoruz; ağacın kendisi sunumda çizilir.

🐍 **Python:** `X.sum()` her öğenin destek sayısı; `sort_values(ascending=False)` büyükten küçüğe sıralar. `kind="stable"`: eşit değerlerin (ekmek ve süt: 6) **önceki sırasını** (alfabetik) korur.

📊 **Konu:** **L sırası:** peynir 7, ekmek 6, süt 6, yağ 2, zeytin 2 (kitaptaki I2, I1, I3, I4, I5 sırası).

In [39]:
sira = X.sum().sort_values(ascending=False, kind="stable")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [40]:
print(sira)

peynir    7
ekmek     6
süt       6
yağ       2
zeytin    2
dtype: int64


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:**
* `list(sira.index)`: sıralanmış öğe adları, liste olarak.
* `sorted(islem, key=L.index)`: işlemi, her öğenin `L` listesindeki **konumuna** göre sıralar (`key`: "neye göre sıralanacak").
* `enumerate(islemler, start=1)`: döngüde her elemanın yanında bir **sıra numarası** da verir (1'den başlayarak).

📊 **Konu:** Her işlemin FP-ağacına **ekleneceği sıra**. Aynı önekle başlayan işlemler ağaçta aynı yolu paylaşır.

In [41]:
L = list(sira.index)
for no, islem in enumerate(islemler, start=1):
    print(f"T{no}00", sorted(islem, key=L.index))

T100 ['peynir', 'ekmek', 'zeytin']
T200 ['peynir', 'yağ']
T300 ['peynir', 'süt']
T400 ['peynir', 'ekmek', 'yağ']
T500 ['ekmek', 'süt']
T600 ['peynir', 'süt']
T700 ['ekmek', 'süt']
T800 ['peynir', 'ekmek', 'süt', 'zeytin']
T900 ['peynir', 'ekmek', 'süt']


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🐍 **Python:** `f"T{no}00"` bir **f-metni**dir: tırnaktan önceki `f`, süslü parantez içindeki değişkenin değerini metne yerleştirir (`no` = 1 ise `T100`).

🔍 **Ara adım:** Önekleri sayalım: 9 işlemin 7'si **peynir** ile başlıyor; ağacın kökünden çıkan "peynir" dalının sayacı 7 olur. Peynirle başlayan işlemlerin 4'ünün ikinci öğesi ekmek: "peynir → ekmek" düğümünün sayacı 4. `Counter` bir listedeki her değerin kaç kez geçtiğini sayar; `[:2]` listenin **ilk iki** elemanı (dilimleme).

In [42]:
from collections import Counter
sirali = [sorted(islem, key=L.index) for islem in islemler]
print(Counter(s[0] for s in sirali))                  # ilk öğe: kökten çıkan dallar
print(Counter(tuple(s[:2]) for s in sirali))          # ilk iki öğe

🐍 **Python:** `fpgrowth`, `apriori` ile **aynı biçimde** çağrılır ve aynı biçimde tablo döndürür.

📊 **Konu:** Aynı veri, aynı eşik: sonuç Apriori'ninkiyle aynı olmalı. İki algoritma **ne bulunacağında** değil, **nasıl bulunduğunda** ayrılır.

In [43]:
sık_fp = fpgrowth(X, min_support=0.2, use_colnames=True)

**Sunumdaki hâli** (tek hücre; slayttaki kodun aynısı):

In [44]:
sira = X.sum().sort_values(ascending=False, kind="stable")    # L sırası
L = list(sira.index)
for no, islem in enumerate(islemler, start=1):
    print(f"T{no}00", sorted(islem, key=L.index))             # FP-ağacına ekleme sırası

sık_fp = fpgrowth(X, min_support=0.2, use_colnames=True)
print(len(sık_fp), set(sık_fp["itemsets"]) == set(sık["itemsets"]))

✅ **Sonucu okuyalım:** FP-Growth da **13** sık küme buldu ve kümeler Apriori'ninkiyle aynı (`True`). İşlemler L sırasına dizilince ortak önekler ortaya çıkıyor: 7 işlem peynirle başlıyor, bunların 4'ü "peynir, ekmek" ile. FP-ağacında bu 7 işlem tek bir daldan, 4'ü de o dalın tek bir alt dalından geçer: veri sıkışır.

## 6. Gerçek veri: Weka'daki supermarket.arff

**Konu.** Bir süpermarketin **4627** alışverişi (Weka ile gelen `supermarket.arff`; Weka'da açtığımız dosyanın aynısı). Her satır bir alışveriş, her sütun bir ürün **bölümü** (ör. `bread and cake`, `fruit`); hücrede `t` (*true*, var) ya da `?` (yok). Son sütun `total`, alışverişin tutarının `low` (düşük) ya da `high` (yüksek) olduğunu söyler.

Dosyayı Weka'nın GitHub deposundan indiriyoruz.

🐍 **Python:** `!wget` bir sistem komutudur: verilen adresteki dosyayı Colab'ın diskine indirir. `-q` mesajları kısaltır, `-nc` (*no clobber*) dosya zaten varsa yeniden indirmez.

In [45]:
!wget -q -nc https://raw.githubusercontent.com/Waikato/weka-3.8/master/wekadocs/data/supermarket.arff

🐍 **Python:**
* `from scipy.io import arff`: `scipy` kütüphanesinin ARFF okuyucusu. ARFF, Weka'nın dosya biçimidir; pandas onu doğrudan okuyamaz.
* `arff.loadarff(...)` **iki değer** döndürür: veri ve dosyanın üst bilgisi (öznitelik tanımları). **Çoklu atama** ile ikisini `veri` ve `ust` adlarına alıyoruz; `ust`'u kullanmayacağız.
* `pd.DataFrame(veri)`: veriyi tabloya çevirir.

📊 **Konu:** 4627 satır, 217 sütun (216 bölüm ve `total`).

In [85]:
from scipy.io import arff
veri, ust = arff.loadarff("supermarket.arff")
sm = pd.DataFrame(veri)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

🔍 **Ara adım:** İlk satırların ilk birkaç sütunu. Değerler `b't'` ve `b'?'` biçiminde: baştaki `b`, değerin metin değil **bayt dizisi** (*bytes*) olduğunu söyler; `scipy` ARFF'teki metinleri böyle okur. Karşılaştırırken biz de `b"t"` yazacağız. `iloc[:3, :6]`: ilk 3 satır, ilk 6 sütun (konuma göre seçim).

In [ ]:
print(sm.shape)
sm.iloc[:3, :6]

🐍 **Python:**
* `sm.drop(columns="total")`: `total` sütunu **dışındaki** bütün sütunlar (tablo değişmez, yeni bir tablo döner).
* `== b"t"`: her hücre için "değer `t` mi?": sonuç True/False tablosu. Bu, 1. bölümdeki işlem tablosunun aynı biçimidir.

📊 **Konu:** Her bölüm bir **öğe**; `t` → True (alışverişte var), `?` → False.

In [ ]:
Xs = sm.drop(columns="total") == b"t"

🐍 **Python:** Tabloya iki yeni sütun ekliyoruz; her biri bir True/False sütunu: `total` yüksek mi, düşük mü.

📊 **Konu:** `total` iki değerli bir **öznitelik**tir. Weka'nın Apriori'si her **öznitelik = değer** çiftini ayrı bir öğe sayar: `total=high` ve `total=low`. Aynısını yapıyoruz ki sonuçlar Weka'yla karşılaştırılabilsin. (Proje 1, B seçeneği bu fikri bütün tabloya uygular: 8. bölüm.)

In [ ]:
Xs["total=high"] = sm["total"] == b"high"
Xs["total=low"] = sm["total"] == b"low"

🔍 **Ara adım:** İşlem tablosunun boyutu ve **sepet büyüklüğü**: `Xs.sum(axis=1)` her **satırdaki** True sayısı (o alışverişteki öğe sayısı; `axis=1`: satır boyunca). `describe()` bu sayıların özetini verir: ortalama, en küçük, çeyrekler, en büyük.

In [ ]:
print(Xs.shape)
print(Xs.sum(axis=1).describe().round(1))

🔍 **Ara adım:** En sık 10 öğe ve destekleri. `mean()` True/False sütununda oranı verir (1. bölüm); `head(10)` ilk 10'u alır.

In [ ]:
print(Xs.mean().sort_values(ascending=False).head(10).round(3))

### Weka'nın varsayılan ayarlarını Python'da taklit etmek

**Konu.** Weka'nın Apriori'si en az desteği **sizden istemez**: 1,0'dan başlar, 0,05 adımlarla (`delta`) düşürür ve en az güveni 0,9 olan **10 kural** (`numRules`) bulunca durur; destek 0,1'in (`lowerBoundMinSupport`) altına inmez. Aynı aramayı yapalım; Weka sunumundaki çıktıyla karşılaştırabilmek için.

🐍 **Python:** **`while` döngüsü:** koşul doğru olduğu sürece bloğu tekrarlar.
* `s = 1.0`: başlangıç desteği.
* `while s >= 0.1:` destek 0,1'in altına inene kadar…
* `if len(kural_s) >= 10: break`: 10 kural bulunduysa `break` döngüden **hemen çıkar**.
* `s = round(s - 0.05, 2)`: desteği 0,05 azaltır. `round(…, 2)`: bilgisayarda ondalık sayılar tam yazılamadığı için (0,15 yerine 0,1499999…) iki basamağa yuvarlıyoruz.
* `len(sık_s) > 0` kontrolü: hiç sık küme yoksa kural üretmeye çalışmayız.
* `fpgrowth` kullandık; `apriori` de aynı sonucu verir.

📊 **Konu:** En az desteği, **istenen sayıda güçlü kural çıkana kadar** düşürmek.

In [ ]:
s = 1.0
while s >= 0.1:
    sık_s = fpgrowth(Xs, min_support=s, use_colnames=True)
    if len(sık_s) > 0:
        kural_s = association_rules(sık_s, metric="confidence", min_threshold=0.9)
        if len(kural_s) >= 10:
            break
    s = round(s - 0.05, 2)

🔍 **Ara adım:** Hangi destekte durdu? Bu destek kaç alışverişe karşılık geliyor, kaç sık küme ve kaç kural var? `int(...)` sayıyı tam sayıya çevirir.

In [ ]:
print(s, int(s * len(Xs)), len(sık_s), len(kural_s))

🐍 **Python:** Okunur sütunlar (`A`, `B`) ve sayılar: Weka, öncülü ve kuralı içeren **alışveriş sayılarını** yazar; biz de destek × N ile sayıya çeviriyoruz (`astype(int)`: tam sayı). `nlargest(10, "confidence")`: güveni en büyük 10 satır; `.copy()` sonucu ayrı bir tablo yapar (sonradan sütun ekleyeceğiz).

📊 **Konu:** Weka'nın listelediği ilk 10 kural; Weka'daki "788 ==> 723 conf:(0.92)" biçimindeki satırlarla karşılaştırın.

In [ ]:
N = len(Xs)
kural_s["A"] = kural_s["antecedents"].apply(yaz)
kural_s["B"] = kural_s["consequents"].apply(yaz)
kural_s["A_sayi"] = (kural_s["antecedent support"] * N).round().astype(int)
kural_s["AB_sayi"] = (kural_s["support"] * N).round().astype(int)
ilk10 = kural_s.nlargest(10, "confidence").copy()

🐍 **Python:** `pd.set_option("display.width", 200)`: uzun satırların alt satıra kaymaması için yazdırma genişliğini artırır. `to_string(index=False)`: tabloyu satır numaraları olmadan yazdırır.

In [ ]:
pd.set_option("display.width", 200)
print(ilk10[["A", "B", "A_sayi", "AB_sayi", "confidence"]].round(2).to_string(index=False))

**Sunumdaki hâli** (tek hücre; slayttaki kodun aynısı):

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/Waikato/weka-3.8/master/wekadocs/data/supermarket.arff
from scipy.io import arff
veri, ust = arff.loadarff("supermarket.arff")
sm = pd.DataFrame(veri)
Xs = sm.drop(columns="total") == b"t"                     # her bölüm bir öğe
Xs["total=high"] = sm["total"] == b"high"                 # öznitelik = değer öğeleri
Xs["total=low"] = sm["total"] == b"low"
print(Xs.shape, round(Xs.sum(axis=1).mean(), 1))          # işlem sayısı, ortalama sepet

s = 1.0                                                   # Weka'nın varsayılan araması
while s >= 0.1:
    sık_s = fpgrowth(Xs, min_support=s, use_colnames=True)
    if len(sık_s) > 0:
        kural_s = association_rules(sık_s, metric="confidence", min_threshold=0.9)
        if len(kural_s) >= 10:
            break
    s = round(s - 0.05, 2)
print(s, int(s * len(Xs)), len(sık_s), len(kural_s))

N = len(Xs)
kural_s["A"] = kural_s["antecedents"].apply(yaz)
kural_s["B"] = kural_s["consequents"].apply(yaz)
kural_s["A_sayi"] = (kural_s["antecedent support"] * N).round().astype(int)
kural_s["AB_sayi"] = (kural_s["support"] * N).round().astype(int)
ilk10 = kural_s.nlargest(10, "confidence").copy()
print(ilk10[["A", "B", "A_sayi", "AB_sayi", "confidence"]].round(2).to_string(index=False))

✅ **Sonucu okuyalım:** Arama en az destek **0,15**'te durdu: 0,15 × 4627 = **694** alışveriş. Bu destekte **2066** sık küme ve güveni 0,9'u aşan **16** kural var; Weka bunların güveni en yüksek 10'unu listeler. İlk kural: {biscuits, frozen foods, fruit, total=high} ⇒ {bread and cake}: öncül **788** alışverişte, kuralın tamamı **723** alışverişte; güven 723 / 788 = **0,92**. Weka'da varsayılan ayarlarla aldığınız çıktı da "Minimum support: 0.15 (694 instances)" der ve aynı 10 kuralı aynı sırayla listeler (Weka FPGrowth da "found 16 rules" der).

**Küçük bir fark:** Weka'nın L(1)…L(6) satırlarının toplamı **2073** sık küme; Python **2066** diyor. Weka 0,15 × 4627 = 694,05'i **694 alışverişe** çevirip "en az 694" diyor; `mlxtend` ise destek ≥ 0,15, yani pratikte **en az 695** alışveriş istiyor. Tam 694 alışverişte geçen 7 küme farkı oluşturuyor (`min_support=694/4627` verilince Python da 2073 buluyor). Weka ile karşılaştırma yaparken eşiği **sayı olarak** eşitleyin.

Dikkat çeken iki şey: on kuralın **hepsinin** sonucu `bread and cake` ve hepsinin öncülünde `total=high` var. Yüksek tutarlı alışverişler büyük sepetlerdir; büyük bir sepette ekmek ve kek bölümünden bir şey olması şaşırtıcı değildir. Bu kuralların gerçekten ilginç olup olmadığını 7. bölümde soracağız.

### Ölçeklenme: Apriori ve FP-Growth'un süresi

**Konu.** En az destek düştükçe sık küme sayısı hızla artar. İki algoritmanın sürelerini üç eşikte ölçelim (Proje 1, A seçeneğindeki ölçeklenme karşılaştırmasının küçük bir örneği).

🐍 **Python:**
* `import time`: zaman ölçme modülü. `time.time()` o anın saniye cinsinden değeri; iki ölçümün farkı geçen süredir.
* `sonuc = []`: boş liste; her deneme bir **sözlük** olarak eklenir (`append`).
* `for ad, islev in [("apriori", apriori), ("fpgrowth", fpgrowth)]:` listede iki **demet** (parantezli çift) var; her turda biri `ad`'a, diğeri `islev`'e atanır. Python'da bir **işlev de bir değerdir**: listeye konup sonra `islev(...)` diye çağrılabilir.
* `pd.DataFrame(sonuc)`: sözlük listesini tabloya çevirir; `pivot` (4. hafta) satırlara eşik, sütunlara algoritma koyar.

📊 **Konu:** Aynı eşikte iki algoritma **aynı sayıda** küme bulmalı; yalnız süreleri farklı olabilir.

In [ ]:
import time
sonuc = []
for ms in [0.2, 0.15, 0.1]:
    for ad, islev in [("apriori", apriori), ("fpgrowth", fpgrowth)]:
        t0 = time.time()
        bulunan = islev(Xs, min_support=ms, use_colnames=True)
        sonuc.append({"min_support": ms, "algoritma": ad,
                      "sure_sn": round(time.time() - t0, 2), "kume_sayisi": len(bulunan)})
olcek = pd.DataFrame(sonuc)

In [ ]:
print(olcek.pivot(index="min_support", columns="algoritma", values="kume_sayisi"))
print(olcek.pivot(index="min_support", columns="algoritma", values="sure_sn"))

✅ **Sonucu okuyalım:** İki algoritma her eşikte **aynı sayıda** sık küme buldu: 0,2'de 640, 0,15'te 2066, 0,1'de **10 282**. Eşiği yarıya indirmek küme sayısını 16 kattan fazla artırdı: en az destek, sonucun boyutunu belirleyen asıl karardır.

Süreler makineden makineye değişir; Colab'da farklı sayılar göreceksiniz. Bu çalıştırmada `mlxtend`'in `apriori`'si `fpgrowth`'tan **daha hızlıydı**. Bu, FP-Growth'un kitaptaki üstünlüğüyle çelişmez: bir algoritmanın kuramsal avantajı ile bir kütüphanedeki **gerçeklemesinin** hızı ayrı şeylerdir (`mlxtend`'in Apriori'si tablo işlemleriyle hızlandırılmıştır; veri ve eşik büyüdükçe tablo da büyür). Projede süre karşılaştırması yaparken hangi gerçeklemeyi kullandığınızı yazın ve sonucu bu ayrımla yorumlayın.

## 7. Kuralların ilginçliği: güven neden yetmez?

**Konu.** Destek ve güven eşiklerini geçen bir kural **güçlü** sayılır; ama güçlü olması **ilginç** olduğu anlamına gelmez. Kitaptaki örnek (Han, Kamber ve Pei, 6.3.1): 10 000 işlemin 6000'inde bilgisayar oyunu, 7500'ünde video, 4000'inde ikisi birlikte. "oyun ⇒ video" kuralının desteği %40, güveni %67: eşikleri geçer. Ama bir müşterinin video alma olasılığı zaten %75; oyun alanlarda bu oran **düşüyor**.

**Lift**, bu karşılaştırmayı yapan ölçüttür:

lift(A, B) = destek(A ∪ B) / (destek(A) × destek(B))

Eşdeğeri:

lift(A, B) = güven(A ⇒ B) / destek(B)

lift = 1: A ve B **bağımsız**; lift > 1: **pozitif ilişki** (birlikte beklenenden sık); lift < 1: **negatif ilişki**. Lift simetriktir: lift(A, B) = lift(B, A).

🐍 **Python:** Dört sayı, dört değişken; sonra destek, güven ve lift formülleri. Bu hücrede kütüphane yok: yalnız bölme ve çarpma.

📊 **Konu:** Kitaptaki oyun–video örneği.

In [ ]:
n, oyun, video, ikisi = 10000, 6000, 7500, 4000
destek_ov = ikisi / n
guven_ov = ikisi / oyun
lift_ov = destek_ov / ((oyun / n) * (video / n))
print(destek_ov, round(guven_ov, 3), video / n, round(lift_ov, 3))

✅ **Sonucu okuyalım:** Destek **0,4**, güven **0,667**, ama video oranı zaten **0,75**: lift **0,889** < 1. Oyun ve video **negatif** ilişkili: oyun alan müşteri video almaya **daha az** eğilimli. Güvene bakıp "oyun alanlara video önerelim" demek yanlış olurdu.

### Ki-kare (χ²)

**Konu.** İkinci bir ilişki ölçütü: 2 × 2 **olumsallık tablosu** (iki değişkenin birlikte görülme sayıları) üzerinden, gözlenen sayıların **bağımsızlık varsayımında beklenen** sayılardan ne kadar saptığı:

χ² = Σ (gözlenen − beklenen)² / beklenen

Beklenen sayı = satır toplamı × sütun toplamı / N. χ² büyükse bağımsızlıktan sapma büyüktür; yönünü (pozitif mi negatif mi) gözlenen ile beklenenin karşılaştırması söyler.

🐍 **Python:**
* `from scipy.stats import chi2_contingency`: istatistik bölümünden ki-kare işlevi.
* `tablo` bir **liste listesi**: 2 satır (oyun var / yok), 2 sütun (video var / yok).
* `chi2_contingency(tablo, correction=False)` dört değer döndürür: χ², p-değeri, serbestlik derecesi, beklenen sayılar. Çoklu atamayla dört ada alıyoruz. `correction=False`: küçük tablolar için yapılan Yates düzeltmesini kapatır (kitaptaki hesap düzeltmesizdir).

📊 **Konu:** Oyun–video olumsallık tablosu: 4000 ikisi, 2000 yalnız oyun, 3500 yalnız video, 500 hiçbiri.

In [ ]:
from scipy.stats import chi2_contingency
tablo = [[4000, 2000],     # oyun var:  video var, video yok
         [3500,  500]]     # oyun yok:  video var, video yok
chi2, p, sd, beklenen = chi2_contingency(tablo, correction=False)

🔍 **Ara adım:** Beklenen sayılar: oyun ve video bağımsız olsaydı "ikisi birlikte" hücresinde 6000 × 7500 / 10 000 = **4500** işlem beklenirdi; gözlenen 4000. Gözlenen < beklenen: **negatif** ilişki.

In [ ]:
print(beklenen)

In [ ]:
print(round(chi2, 1))

✅ **Sonucu okuyalım:** χ² = **555,6**: bağımsızlıktan güçlü bir sapma; "ikisi birlikte" hücresinde gözlenen (4000) beklenenden (4500) az olduğu için sapma **negatif** yönde. Lift ile aynı sonuç.

### Gerçek veride güven ve lift

**Konu.** 6. bölümdeki supermarket kurallarının güveni 0,9'un üstündeydi. Lift ne diyor? Bütün kuralların sonucu `bread and cake`; bu öğe alışverişlerin büyük kısmında zaten var.

🐍 **Python:** `consequent support`: kuralın **sonucunun** (B) desteği; `association_rules` tablosunda hazır.

In [ ]:
print(ilk10[["A", "confidence", "consequent support", "lift"]].round(3).to_string(index=False))

✅ **Sonucu okuyalım:** Güven 0,905–0,918, ama `bread and cake` alışverişlerin zaten **%72**'sinde var. Lift 1,26–1,28: ilişki pozitif ama **zayıf**. Güvenin büyük kısmı, sonucun **popülerliğinden** geliyor: rastgele bir alışverişte bile %72 olasılıkla ekmek ve kek bölümünden bir şey var.

### Null işlemler: lift ve χ² neden yanıltabilir?

**Konu.** **Null işlem**: incelenen öğelerin **hiçbirini** içermeyen işlem. Gerçek veride çoğu işlem, iki belirli ürün için null'dur (bir markette her ürün, alışverişlerin küçük bir kısmında geçer). Lift ve χ², null işlem sayısına **duyarlıdır**: A ve B'nin birlikte görülme biçimi hiç değişmese de, araya ilgisiz işlemler eklenince değerleri değişir.

Deney: 1. bölümdeki 9 işlemlik sepete **boş** işlemler ekleyelim ve "ekmek ⇒ süt" kuralına bakalım.

🐍 **Python:**
* `def kural_olc(tablo, a, b):` bir **işlev tanımı**: adı `kural_olc`, üç **parametre** alır. Gövde girintili yazılır; `return` sonucu geri verir.
* İçeride 1. bölümdeki kalıplar: `tablo[a].mean()` A'nın desteği; `tablo[[a, b]].all(axis=1).mean()` ikisinin birlikte desteği.
* `return` bir **sözlük** döndürür: ölçütlerin adları ve değerleri.

📊 **Konu:** Bir kuralın destek, güven, lift ve Kulczynski değerlerini tablodan hesaplayan küçük bir işlev. Kulczynski'yi birazdan tanımlayacağız; şimdilik formülü:

Kulc(A, B) = ½ × (güven(A ⇒ B) + güven(B ⇒ A))

In [ ]:
def kural_olc(tablo, a, b):
    da, db = tablo[a].mean(), tablo[b].mean()
    dab = tablo[[a, b]].all(axis=1).mean()
    return {"N": len(tablo), "destek": dab, "guven": dab / da,
            "lift": dab / (da * db), "kulc": 0.5 * (dab / da + dab / db)}

🔍 **Ara adım:** İşlevi bir kez çağıralım: 9 işlemde ekmek ⇒ süt. `pd.Series(...)` sözlüğü tek sütunluk bir tabloya çevirir; böylece `round(3)` ile yuvarlayıp düzgün yazdırabiliriz. Lift **1,0**: ekmek ve süt bu 9 işlemde **bağımsız** (ekmek oranı 6/9, süt oranı 6/9, birlikte 4/9 = 6/9 × 6/9).

In [ ]:
print(pd.Series(kural_olc(X, "ekmek", "süt")).round(3))

🐍 **Python:**
* `pd.DataFrame(False, index=range(k), columns=X.columns)`: `k` satırlı, `X` ile aynı sütunlu, **bütün hücreleri False** olan bir tablo: k tane boş işlem.
* `pd.concat([X, ...], ignore_index=True)`: iki tabloyu **alt alta** ekler; `ignore_index=True` satır numaralarını baştan verir.
* Döngü üç durumu dener: 0, 90 ve 990 boş işlem. Her sonuç (sözlük) bir listeye eklenir, sonra tabloya çevrilir.

📊 **Konu:** Ekmek ve süt içeren işlemler **hiç değişmiyor**; yalnız ilgisiz (null) işlemlerin sayısı artıyor.

In [ ]:
satirlar = []
for k in [0, 90, 990]:
    bos = pd.DataFrame(False, index=range(k), columns=X.columns)
    X_null = pd.concat([X, bos], ignore_index=True)
    satirlar.append(kural_olc(X_null, "ekmek", "süt"))
print(pd.DataFrame(satirlar).round(3))

✅ **Sonucu okuyalım:** Ekmek ve süt içeren işlemler hiç değişmedi; yalnız **ilgisiz** işlemler eklendi. Güven **0,667**'de sabit kaldı, Kulczynski de **0,667**'de sabit. Lift ise **1,0**'dan (bağımsız) **11**'e ve **111**'e çıktı: ilişki değişmediği hâlde lift "çok güçlü pozitif ilişki" diyor. Destek de düştü (0,444 → 0,004), çünkü paydası N. Gerçek veride null işlemler hep çoktur; bu yüzden lift (ve χ²) çoğu zaman yanıltıcı biçimde büyük çıkar.

### Null-değişmez ölçütler

**Konu.** Bir ölçüt, değeri null işlem sayısından **etkilenmiyorsa** null-değişmezdir. Kitapta (6.3.3) ve Wu, Chen ve Han (2010) makalesinde incelenen dört ölçüt; hepsi yalnız iki **koşullu olasılığa** dayanır: P(A|B) = destek(A ∪ B) / destek(B) ve P(B|A) = destek(A ∪ B) / destek(A). Bu iki olasılıkta null işlemler pay ve paydada **hiç geçmez**.

all_conf(A, B) = destek(A ∪ B) / max(destek(A), destek(B))

max_conf(A, B) = max(P(A|B), P(B|A))

Kulc(A, B) = ½ × (P(A|B) + P(B|A))

cosine(A, B) = destek(A ∪ B) / √(destek(A) × destek(B))

Hepsi 0 ile 1 arasındadır; 0,5 civarı **nötr**, 1'e yakın güçlü pozitif, 0'a yakın negatif ilişki demektir. all_conf = min(P(A|B), P(B|A)) olduğu için Wu, Chen ve Han bu dördünü aynı iki olasılığın **ortalamaları** olarak birleştirir: en küçüğü (all_conf), geometrik ortalama (cosine), aritmetik ortalama (Kulc), en büyüğü (max_conf). Bu yüzden her zaman:

all_conf ≤ cosine ≤ Kulc ≤ max_conf

İki koşullu olasılık çok farklıysa (bir öğe çok yaygın, öteki seyrek), ölçütler birbirinden ayrılır. Bu **dengesizliği** ayrıca ölçeriz: **dengesizlik oranı** (IR, *imbalance ratio*):

IR(A, B) = |destek(A) − destek(B)| / (destek(A) + destek(B) − destek(A ∪ B))

IR = 0: dengeli; IR 1'e yaklaştıkça dengesiz. Kitabın önerisi: **Kulczynski ile IR'yi birlikte** raporlamak.

🐍 **Python:**
* `def olcutler(ab, yalniz_b, yalniz_a, hicbiri):` 2 × 2 olumsallık tablosunun dört hücresini alır: A ve B birlikte, yalnız B, yalnız A, hiçbiri (null).
* `a`, `b`: A ve B'nin toplam sayıları; destek yerine sayıları kullanabiliriz, çünkü N formüllerde sadeleşir (lift ve χ² hariç; onlarda N kullanılır).
* `max(...)`, `min(...)`, `abs(...)`: en büyük, en küçük, mutlak değer. `** 0.5`: karekök (`**` üs alma).
* `chi2_contingency(...)[0]`: döndürülen dört değerin **ilki** (χ²); `[0]` ilk elemanı seçer.

📊 **Konu:** Altı ölçüt ve IR tek işlevde.

In [ ]:
def olcutler(ab, yalniz_b, yalniz_a, hicbiri):
    N = ab + yalniz_b + yalniz_a + hicbiri
    a, b = ab + yalniz_a, ab + yalniz_b                  # A ve B'nin toplam sayısı
    return {
        "lift":     round((ab / N) / ((a / N) * (b / N)), 2),
        "chi2":     round(chi2_contingency([[ab, yalniz_a], [yalniz_b, hicbiri]], correction=False)[0]),
        "all_conf": round(ab / max(a, b), 2),
        "max_conf": round(max(ab / a, ab / b), 2),
        "kulc":     round(0.5 * (ab / a + ab / b), 2),
        "cosine":   round(ab / (a * b) ** 0.5, 2),
        "IR":       round(abs(a - b) / (a + b - ab), 2),
    }

🔍 **Ara adım:** İşlevi oyun–video örneğinde deneyelim (A = oyun, B = video): 4000 ikisi, 3500 yalnız video, 2000 yalnız oyun, 500 hiçbiri. Lift ve χ² yukarıdaki değerlerle aynı çıkmalı.

In [ ]:
print(olcutler(4000, 3500, 2000, 500))

**Konu.** Kitabın altı veri kümesi (Han, Kamber ve Pei, 6.3.3; süt *m* ve kahve *c*): her biri dört sayıyla verilir: mc (ikisi), m̄c (yalnız kahve), mc̄ (yalnız süt), m̄c̄ (hiçbiri).

🐍 **Python:**
* `veri_kumeleri` bir **sözlük**: anahtar veri kümesinin adı, değer dört sayılık bir **demet**.
* `.items()`: sözlüğün (anahtar, değer) çiftlerini dolaşır.
* `olcutler(*sayilar)`: baştaki `*`, demeti **açar**: dört sayı, işleve dört ayrı parametre olarak gider.
* Sözlük üreteciyle `{ad: olcutler(...)}` kurup `pd.DataFrame(...).T` ile tabloya çeviriyoruz (`.T`: satırlar veri kümeleri olsun diye çevirme, 4. hafta).

In [ ]:
veri_kumeleri = {
    "D1": (10000, 1000, 1000, 100000),
    "D2": (10000, 1000, 1000, 100),
    "D3": (100, 1000, 1000, 100000),
    "D4": (1000, 1000, 1000, 100000),
    "D5": (1000, 100, 10000, 100000),
    "D6": (1000, 10, 100000, 100000),
}
karsilastirma = pd.DataFrame({ad: olcutler(*sayilar) for ad, sayilar in veri_kumeleri.items()}).T
print(karsilastirma)

✅ **Sonucu okuyalım:** * **D1 ve D2** yalnız null işlem sayısında farklı (m̄c̄: 100 000 ile 100). Lift **9,26**'dan **1,00**'e, χ² **90 557**'den **0**'a düşüyor: biri "çok güçlü ilişki", öteki "bağımsız" diyor. Dört null-değişmez ölçüt ikisinde de aynı: **0,91** (süt alanların çoğu kahve, kahve alanların çoğu süt alıyor: güçlü pozitif).
* **D3:** lift 8,44 ve χ² 670 pozitif bir ilişki gösteriyor; null-değişmez ölçütler **0,09**: süt alanların yalnız %9'u kahve alıyor (ve tersi). Lift'i büyüten yine 100 000 null işlem.
* **D4:** null-değişmez ölçütler **0,5**: nötr. Lift ise **25,75** ile tablonun en büyüğü.
* **D5 ve D6: dengesiz** durumlar. Kahve alanların çoğu süt alıyor ama süt alanların çok azı kahve alıyor. Dört ölçüt **ayrışıyor** (D6'da all_conf 0,01, max_conf 0,99); sıralama her zaman all_conf ≤ cosine ≤ Kulc ≤ max_conf. Kulczynski ikisinde de **0,5** (nötr) diyor; ama tek başına D4'teki "gerçekten nötr" durumdan ayırt edilemez. Bu ayrımı **IR** yapıyor: D4'te 0, D5'te **0,89**, D6'da **0,99**.

Sonuç (kitabın ve Wu, Chen ve Han'ın önerisi): **Kulczynski ile IR'yi birlikte** raporlayın. Kulczynski ilişkinin yönünü ve gücünü, IR iki tarafın dengesini söyler.

### Supermarket kurallarında Kulczynski ve IR

🐍 **Python:** `association_rules` tablosunda `kulczynski` sütunu hazırdır. IR'yi kendimiz ekliyoruz: `antecedent support` (A'nın desteği), `consequent support` (B'nin desteği) ve `support` (birlikte) sütunlarıyla, formülün aynısı. `abs` bir sütuna uygulanınca her değerin mutlak değerini alır.

📊 **Konu:** Güveni 0,9'u aşan ilk 10 kural, null-değişmez gözle.

In [ ]:
da, db, dab = ilk10["antecedent support"], ilk10["consequent support"], ilk10["support"]
ilk10["IR"] = (da - db).abs() / (da + db - dab)
print(ilk10[["A", "confidence", "lift", "kulczynski", "IR"]].round(3).to_string(index=False))

✅ **Sonucu okuyalım:** Kulczynski **0,56–0,58**: 0,5'e yakın, yani zayıf pozitif. IR **0,69–0,76**: güçlü bir **dengesizlik**: öncüller alışverişlerin yaklaşık %16–21'inde, sonuç (`bread and cake`) %72'sinde. Öncülü içeren alışverişlerin çoğunda ekmek ve kek var (güven 0,9), ama ekmek ve kek içeren alışverişlerin çoğunda öncül yok. Güveni 0,9'u aşan bu "güçlü" kurallar, null-değişmez gözle bakınca, **yaygın bir ürünün yaygın olmasını** anlatıyor.

**Sunumdaki hâli** (tek hücre; slayttaki kodun aynısı):

In [ ]:
from scipy.stats import chi2_contingency

def olcutler(ab, yalniz_b, yalniz_a, hicbiri):
    N = ab + yalniz_b + yalniz_a + hicbiri
    a, b = ab + yalniz_a, ab + yalniz_b
    return {
        "lift":     round((ab / N) / ((a / N) * (b / N)), 2),
        "chi2":     round(chi2_contingency([[ab, yalniz_a], [yalniz_b, hicbiri]], correction=False)[0]),
        "all_conf": round(ab / max(a, b), 2),
        "max_conf": round(max(ab / a, ab / b), 2),
        "kulc":     round(0.5 * (ab / a + ab / b), 2),
        "cosine":   round(ab / (a * b) ** 0.5, 2),
        "IR":       round(abs(a - b) / (a + b - ab), 2),
    }

print(olcutler(4000, 3500, 2000, 500))                    # oyun ⇒ video
veri_kumeleri = {"D1": (10000, 1000, 1000, 100000), "D2": (10000, 1000, 1000, 100),
                 "D3": (100, 1000, 1000, 100000),   "D4": (1000, 1000, 1000, 100000),
                 "D5": (1000, 100, 10000, 100000),  "D6": (1000, 10, 100000, 100000)}
print(pd.DataFrame({ad: olcutler(*sayilar) for ad, sayilar in veri_kumeleri.items()}).T)

## 8. Proje 1, 3. bölüm: birliktelik analizi

**Konu.** Yönergede birliktelik analizi iki seçeneklidir; veri değiştirilmez, seçim gerekçesiyle defterde yazılır. Bölümün makalesi Wu, Chen ve Han (2010): 7. bölümdeki null-değişmezlik ve "ortalamalar" çerçevesi bu makaleden gelir.

* **Seçenek A** (verinizde sepet, sipariş, oturum ya da ziyaret bilgisi varsa): bu defterin 6. ve 7. bölümleri. İşlem tanımınızı yazın (bir işlem bir sipariş mi, bir oturum mu, bir müşteri-gün mü?) ve farklı bir tanımın sonucu nasıl değiştireceğini tartışın. Ölçeklenme için 6. bölümdeki süre döngüsünü, null-invariance testi için 7. bölümdeki boş işlem deneyinin mantığını kullanın (yönergede: 200 ilgisiz nadir öğe).
* **Seçenek B** (her satır bir kişi ya da olay): tabloyu işleme çevirirsiniz. Aşağıda credit-g verisiyle (4. haftanın verisi) bu dönüşümün nasıl yapıldığını görüyoruz.

### Seçenek B: tabloyu işleme çevirmek (credit-g)

**Konu.** Her kayıt bir **işlem**, her "öznitelik = değer" çifti bir **öğe** olur: ör. `{checking_status=<0, age=19-30, class=bad}`. Sayısal öznitelikler önce **ayrıklaştırılır** (3. hafta); yaş için 4. haftadaki **kavram hiyerarşisinin** aynısını (yaş → yaş grubu) kullanıyoruz. Bu, 6. bölümde `total=high` için yaptığımızın bütün tabloya uygulanmasıdır.

🐍 **Python:** Aynı ARFF okuma kalıbı (6. bölüm). 1000 başvuru, 21 sütun.

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/Waikato/weka-3.8/master/wekadocs/data/credit-g.arff
veri, ust = arff.loadarff("credit-g.arff")
kredi = pd.DataFrame(veri)

🐍 **Python:** `apply`, bu kez **her sütuna** bir işlev uygular. İşlev bir `lambda`:
* `s.dtype == object`: sütun metin (bayt) sütunu mu? (Sayısal sütunların türü `float64`'tür.)
* `... if ... else s`: **koşullu ifade**: koşul doğruysa ilk değer, değilse `s`'nin kendisi.
* `s.str.decode("utf-8")`: baytları düz metne çevirir (`b'good'` → `'good'`).

📊 **Konu:** Metin sütunlarını okunur hâle getiriyoruz; sayısal sütunlar değişmez.

In [ ]:
kredi = kredi.apply(lambda s: s.str.decode("utf-8") if s.dtype == object else s)

🔍 **Ara adım:** Kullanacağımız sütunlara bakalım. Kötü kredi oranı 300 / 1000 = 0,3 (4. hafta).

In [ ]:
print(kredi[["checking_status", "duration", "credit_amount", "age", "class"]].head())
print(kredi["class"].value_counts())

🐍 **Python:**
* `pd.cut(sütun, bins=..., labels=...)`: sayıları **sınırları sizin verdiğiniz** aralıklara böler (4. hafta). Yaş için 4. haftadaki sınırlar; süre (ay) için 12 ve 24.
* `pd.qcut(sütun, 3, labels=...)`: sayıları **eşit kayıt sayılı** 3 gruba böler (çeyreklik mantığı; sınırları veri belirler).

📊 **Konu:** Ayrıklaştırma sınırları birer **karardır** ve projede gerekçelendirilir: yaş ve süre için anlamlı eşikler (yıl, ay), tutar için eşit sıklıklı gruplar.

In [ ]:
kredi["age"] = pd.cut(kredi["age"], bins=[18, 30, 45, 75], labels=["19-30", "31-45", "46-75"])
kredi["duration"] = pd.cut(kredi["duration"], bins=[0, 12, 24, 72], labels=["<=12", "13-24", "25-72"])
kredi["credit_amount"] = pd.qcut(kredi["credit_amount"], 3, labels=["düşük", "orta", "yüksek"])

🔍 **Ara adım:** Her grupta kaç kayıt var? `qcut` gruplarının yaklaşık eşit olduğuna dikkat edin.

In [ ]:
print(kredi["age"].value_counts().sort_index())
print(kredi["duration"].value_counts().sort_index())
print(kredi["credit_amount"].value_counts().sort_index())

🐍 **Python:**
* `secilen`: kullanacağımız 9 sütunun adları (bir liste).
* `kredi[secilen].astype(str)`: bütün değerleri metne çevirir (kategorik ve metin sütunları aynı türde olsun).
* `pd.get_dummies(..., prefix_sep="=")`: **tek-sıcak kodlama** (3. hafta): her (sütun, değer) çifti için bir True/False sütunu; sütun adı `sütun=değer` biçiminde olur.
* `astype(bool)`: True/False türüne çevirir (`mlxtend` bunu ister).

📊 **Konu:** **İşlem tablosu**: her satır bir başvuru, her sütun bir "öznitelik = değer" öğesi. Her satırda tam 9 öğe True'dur (her öznitelikten bir değer).

In [ ]:
secilen = ["checking_status", "credit_history", "purpose", "savings_status",
           "duration", "credit_amount", "age", "housing", "class"]
T = pd.get_dummies(kredi[secilen].astype(str), prefix_sep="=").astype(bool)

🔍 **Ara adım:** Tablonun boyutu, ilk birkaç öğe sütunu ve her satırdaki öğe sayısı (hepsi 9 olmalı).

In [ ]:
print(T.shape)
print(T.columns[:6].tolist())
print(T.sum(axis=1).unique())

### Hedefe yönelik kurallar

🐍 **Python:**
* `apriori(T, min_support=0.05, use_colnames=True, max_len=4)`: `max_len=4`, en fazla 4 öğeli kümeleri arar (öncülde en fazla 3 öğe, sonuçta hedef).
* `kural_b["consequents"] == frozenset(["class=bad"])`: sonucu **yalnız** `class=bad` olan kurallar (mantıksal filtre).
* `.copy()`: süzülmüş tabloyu **ayrı bir kopya** yapar; böylece sonradan ona sütun eklerken pandas uyarı vermez.

📊 **Konu:** **Hedefe yönelik kurallar:** sonucu hedef değişken olan kurallar ("hangi özellik birleşimleri kötü krediyle birlikte görülüyor?"). Güven eşiği 0,5: kötü kredi oranı 0,3 olduğu için, güveni 0,5'i aşan bir kural bu oranın belirgin üstündedir.

In [ ]:
sık_b = apriori(T, min_support=0.05, use_colnames=True, max_len=4)
kural_b = association_rules(sık_b, metric="confidence", min_threshold=0.5)
hedef = kural_b[kural_b["consequents"] == frozenset(["class=bad"])].copy()

🔍 **Ara adım:** Kaç sık küme, kaç kural, kaçının sonucu `class=bad`?

In [ ]:
print(len(sık_b), len(kural_b), len(hedef))

In [ ]:
hedef["A"] = hedef["antecedents"].apply(yaz)
print(hedef.sort_values("confidence", ascending=False)
      [["A", "support", "confidence", "lift", "kulczynski"]].round(3).head(8).to_string(index=False))

✅ **Sonucu okuyalım:** Destek ≥ 0,05 ve güven ≥ 0,5 ile **12** hedefe yönelik kural var (sık küme 993, toplam kural 1461). En güçlüsü: {duration=25-72, savings_status=<100} ⇒ class=bad: başvuruların %7,6'sında birlikte görülüyor, güven **0,559** (genel kötü oranı 0,3), lift **1,86**. İlk sekiz kuralda aynı öğeler tekrar ediyor: **checking_status=<0** (hesapta eksi bakiye), **savings_status=<100** (düşük birikim), **duration=25-72** (uzun vade), **age=19-30**. Yönergedeki soru: bu öğeler, Proje 1 modelinizin önemli öznitelikleriyle örtüşüyor mu? (4. haftada genç başvuranlarda kötü oranının daha yüksek olduğunu görmüştük.)

Kulczynski değerleri 0,35–0,45: kötü kredilerin çoğu bu öncülleri **içermiyor** (P(A | kötü) küçük). Bir kural, kötü kredilerin yalnız bir alt grubunu tarif eder.

### Nadir kategoriler: lift ile Kulczynski'nin farkı

**Konu.** Yönergenin B seçeneğinde istenen karşılaştırma: tek öncüllü kurallar (A ⇒ class=bad), daha düşük bir destek eşiğiyle (0,02: en az 20 başvuru), böylece **nadir** kategoriler de kurallara girer. Lift'e göre ve Kulczynski'ye göre ilk 5 kural aynı mı?

🐍 **Python:** `max_len=2`: en fazla 2 öğeli kümeler (bir öncül, bir sonuç). `nlargest(5, "lift")`: lift'i en büyük 5 kural. `antecedent support × 1000`: öncülün kaç başvuruda geçtiği (`round()` en yakın tam sayıya yuvarlar). IR sütunu, supermarket kurallarındaki hesabın aynısı.

In [ ]:
sık_t = apriori(T, min_support=0.02, use_colnames=True, max_len=2)
kural_t = association_rules(sık_t, metric="confidence", min_threshold=0.0)
tekli = kural_t[kural_t["consequents"] == frozenset(["class=bad"])].copy()
tekli["A"] = tekli["antecedents"].apply(yaz)
tekli["A_sayi"] = (tekli["antecedent support"] * len(T)).round().astype(int)
da, db, dab = tekli["antecedent support"], tekli["consequent support"], tekli["support"]
tekli["IR"] = (da - db).abs() / (da + db - dab)

In [ ]:
sutunlar = ["A", "A_sayi", "confidence", "lift", "kulczynski", "IR"]
print(tekli.nlargest(5, "lift")[sutunlar].round(3).to_string(index=False))
print()
print(tekli.nlargest(5, "kulczynski")[sutunlar].round(3).to_string(index=False))

✅ **Sonucu okuyalım:** İki sıralama **farklı kuralları** öne çıkarıyor:
* **Lift'e göre** ilk sırada **nadir** kategoriler: credit_history=no credits/all paid (**40** başvuru, lift 2,08), credit_history=all paid (**49**), purpose=education (**50**). Güven yüksek ama 40–50 kayda dayanıyor; IR 0,76–0,83: kötü kredilerin çok küçük bir kısmını açıklıyorlar.
* **Kulczynski'ye göre** ilk sırada **yaygın** kategoriler: savings_status=<100 (**603** başvuru), housing=own (713). Kulczynski'nin P(A | kötü) yarısı, yaygın bir öncülde zaten büyüktür. housing=own'un lift'i **0,87**: 1'in altında, yani kötü krediyle **negatif** ilişkili; yine de Kulczynski listesinde. Bu listedeki değerler 0,43–0,54: hepsi **nötre yakın**.
* İki listede de olan tek kural **checking_status=<0** ⇒ class=bad: 274 başvuru, güven 0,49, lift 1,64, Kulczynski 0,47 ve IR **0,06** (dengeli). Hem lift'te hem Kulczynski'de üstlerde ve dengeli: en güvenilir bulgu bu.

Ders: tek bir ölçüte göre "en iyi 5 kural" listesi, ölçütün **neyi ödüllendirdiğini** yansıtır. Lift nadir öncülleri, Kulczynski yaygın öncülleri öne çıkarabilir; IR ve kayıt sayısı (`A_sayi`) birlikte raporlanmalıdır.

**Sunumdaki hâli** (tek hücre; slayttaki kodun aynısı):

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/Waikato/weka-3.8/master/wekadocs/data/credit-g.arff
veri, ust = arff.loadarff("credit-g.arff")
kredi = pd.DataFrame(veri)
kredi = kredi.apply(lambda s: s.str.decode("utf-8") if s.dtype == object else s)

kredi["age"] = pd.cut(kredi["age"], bins=[18, 30, 45, 75], labels=["19-30", "31-45", "46-75"])
kredi["duration"] = pd.cut(kredi["duration"], bins=[0, 12, 24, 72], labels=["<=12", "13-24", "25-72"])
kredi["credit_amount"] = pd.qcut(kredi["credit_amount"], 3, labels=["düşük", "orta", "yüksek"])
secilen = ["checking_status", "credit_history", "purpose", "savings_status",
           "duration", "credit_amount", "age", "housing", "class"]
T = pd.get_dummies(kredi[secilen].astype(str), prefix_sep="=").astype(bool)   # öznitelik = değer

sık_t = apriori(T, min_support=0.02, use_colnames=True, max_len=2)            # tek öncüllü kurallar
kural_t = association_rules(sık_t, metric="confidence", min_threshold=0.0)
tekli = kural_t[kural_t["consequents"] == frozenset(["class=bad"])].copy()      # hedefe yönelik
tekli["A"] = tekli["antecedents"].apply(yaz)
tekli["A_sayi"] = (tekli["antecedent support"] * len(T)).round().astype(int)
da, db, dab = tekli["antecedent support"], tekli["consequent support"], tekli["support"]
tekli["IR"] = (da - db).abs() / (da + db - dab)
sutunlar = ["A", "A_sayi", "confidence", "lift", "kulczynski", "IR"]
print(tekli.nlargest(5, "lift")[sutunlar].round(3).to_string(index=False))
print(tekli.nlargest(5, "kulczynski")[sutunlar].round(3).to_string(index=False))

### Weka ile kontrol (her iki seçenek)

Yönerge, aynı eşiklerde Weka Apriori çıktısının Python ile karşılaştırılmasını ister. A seçeneğinde 6. bölümdeki gibi Weka'nın kural listesini, Python'daki öncül ve kural sayılarıyla (`A_sayi`, `AB_sayi`) satır satır eşleştirin. B seçeneğinde ayrıklaştırılmış tabloyu ARFF olarak kaydedip Weka'da açabilirsiniz; Weka'nın Apriori'si nominal öznitelikleri doğrudan "öznitelik = değer" öğelerine çevirir. Weka'da `minMetric`, `lowerBoundMinSupport` ve `numRules` ayarlarını Python'daki eşiklerle aynı yapmayı unutmayın.

## Bu defterde neler yaptık?

| Bölüm | Kavram | Kod |
|---|---|---|
| 1 | İşlem tablosu, destek, güven | `TransactionEncoder`, `all(axis=1)`, `mean()` |
| 2 | Sık öğe kümeleri, Apriori özelliği, aday üretme ve budama | `apriori`, `combinations`, `all(...)` |
| 3 | Kapalı ve maksimal sık kümeler | küme karşılaştırması `<`, `fpmax` |
| 4 | Sık kümelerden kural üretme | `association_rules` |
| 5 | FP-Growth, L sırası ve önekler | `fpgrowth`, `sorted(key=...)` |
| 6 | Gerçek veri, Weka'nın varsayılan araması, ölçeklenme | `loadarff`, `while`, `time` |
| 7 | Lift, χ², null işlemler, null-değişmez ölçütler, IR | `def`, `chi2_contingency` |
| 8 | Proje 1, 3. bölüm: A ve B seçenekleri | `pd.cut`, `qcut`, `get_dummies` |

Bağlantılar: **en az destek**, 4. haftadaki buzdağı küpünün eşiğiyle aynı fikirdir (yalnız yeterince sık olanı sakla); **ayrıklaştırma** ve **kavram hiyerarşisi** (3. ve 4. hafta), tablo verisini işlem verisine çevirmenin yoludur.